# Premier modèle avec scikit-learn

Avant d'attaquer un vrai algorithme (l'arbre de décision dans le notebook suivant), on prend un moment pour comprendre la **convention** de scikit-learn — le "contrat" auquel obéissent **tous** ses modèles, des plus simples (régression linéaire) aux plus complexes (XGBoost, réseaux de neurones via scikit-learn).

L'idée : une fois ce contrat compris, vous saurez utiliser **n'importe quel** modèle scikit-learn sans avoir à relire la documentation à chaque fois.

## Le contrat scikit-learn en une image

Tout modèle scikit-learn est une **boîte avec deux boutons** :

- `.fit(X, y)` → **apprendre** à partir d'exemples (X = ce qu'on observe, y = la réponse attendue)
- `.predict(X_nouveaux)` → **prédire** la réponse pour de nouvelles observations

Plus un troisième bouton utilitaire :

- `.score(X, y)` → **évaluer** la qualité des prédictions sur des données dont on connaît déjà la vraie réponse

Que ce soit un arbre de décision, une régression linéaire ou un Random Forest, le pattern est **toujours** le même. C'est ce qui rend scikit-learn si agréable à utiliser.

## Charger un dataset

Scikit-learn embarque quelques petits datasets « jouets » prêts à l'emploi pour s'entraîner. On va utiliser le plus célèbre : **Iris** — 150 fleurs d'iris, 4 mesures par fleur (longueur/largeur du sépale et du pétale), et 3 espèces à reconnaître.

In [1]:
from sklearn.datasets import load_iris

iris = load_iris()

Le résultat est un objet « Bunch » qui ressemble à un dictionnaire. Les champs utiles :

In [2]:
# Les attributs principaux d'un dataset sklearn
print("data.shape          :", iris.data.shape)          # 150 lignes, 4 colonnes
print("target.shape        :", iris.target.shape)        # 150 labels
print("feature_names       :", iris.feature_names)       # les 4 mesures
print("target_names        :", iris.target_names)        # les 3 espèces

data.shape          : (150, 4)
target.shape        : (150,)
feature_names       : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
target_names        : ['setosa' 'versicolor' 'virginica']


In [3]:
# Les 5 premières lignes de data : 4 nombres par fleur
iris.data[:5]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2]])

In [4]:
# Les 5 premiers labels : 0 = setosa, 1 = versicolor, 2 = virginica
iris.target[:5]

array([0, 0, 0, 0, 0])

## X et y : la convention sklearn

Tout au long de scikit-learn, on retrouve **deux variables canoniques** :

| Variable | Forme | Contenu |
|----------|-------|---------|
| **`X`** (majuscule) | matrice 2D `(n_samples, n_features)` | les **caractéristiques** observées |
| **`y`** (minuscule) | vecteur 1D `(n_samples,)` | la **cible** à prédire |

**À retenir** :

- `X` est toujours **2D** (matrice), même quand il n'y a qu'une seule feature — auquel cas il faut quand même `X.shape = (n, 1)` et non `(n,)`.
- `y` est **1D** en apprentissage supervisé classique.
- La majuscule de `X` est une convention mathématique (les matrices s'écrivent en majuscule) et `y` en minuscule (les vecteurs).

In [5]:
X = iris.data       # 150 × 4
y = iris.target     # 150

print("X :", X.shape, X.dtype)
print("y :", y.shape, y.dtype)

X : (150, 4) float64
y : (150,) int64


## Découper en train et test : pourquoi et comment

On ne va pas entraîner le modèle sur **toutes** les données et l'évaluer sur ces mêmes données. Pourquoi ?

```{admonition} 💡 L'analogie de l'étudiant qui révise
:class: tip
Imaginez un étudiant qui révise pour un examen en faisant les exercices d'un livre. Si l'examen final est composé **exactement** des mêmes exercices, l'étudiant peut tricher : il les a appris **par cœur** sans vraiment comprendre.

Pour évaluer s'il a réellement appris, il faut lui poser des questions qu'il n'a **jamais vues**.

C'est exactement la même logique en ML : on garde une partie des données **à part** (le "test set") qu'on ne montre au modèle qu'au moment de l'évaluation.
```

Scikit-learn fournit la fonction `train_test_split` pour faire ce découpage en une ligne :

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,     # 25 % en test, 75 % en apprentissage
    random_state=42,    # fige le tirage aléatoire pour la reproductibilité
)

print("Train :", X_train.shape, y_train.shape)
print("Test  :", X_test.shape,  y_test.shape)

Train : (112, 4) (112,)
Test  : (38, 4) (38,)


Le paramètre `random_state` mérite un mot : la fonction tire au sort **aléatoirement** quelles lignes vont en train et lesquelles en test. Pour que vos résultats soient **reproductibles** (et comparables avec ceux d'un collègue), on fige cette graine. Le chiffre lui-même n'a aucune importance — c'est juste une étiquette pour le tirage.

## Un premier modèle : le « modèle bête » (DummyClassifier)

Avant de sortir l'artillerie lourde, on commence **toujours** par un modèle aussi simple que possible. Pourquoi ? Pour avoir un **point de comparaison** — une *baseline* — au-dessus duquel tout vrai modèle doit se situer. Sinon on ne sait pas si une accuracy de 80 % est bonne ou pas.

Le `DummyClassifier` est exprès **stupide** : avec la stratégie `"most_frequent"`, il prédit toujours la classe **majoritaire** du jeu d'entraînement. Pas d'apprentissage, pas d'intelligence.

In [7]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")

# Apprentissage
baseline.fit(X_train, y_train)

# Prédiction sur les 5 premières fleurs de test
baseline.predict(X_test[:5])

array([1, 1, 1, 1, 1])

In [8]:
# Score = accuracy par défaut en classification (proportion de bonnes réponses)
score_baseline = baseline.score(X_test, y_test)
print(f"Baseline (classe majoritaire) : {score_baseline:.2%}")

Baseline (classe majoritaire) : 28.95%


Sur Iris (3 classes équilibrées), prédire toujours la classe majoritaire donne environ **33 %**. C'est notre **plancher** : tout vrai modèle doit faire mieux que ça.

## Un modèle un peu moins bête : KNeighborsClassifier

Toujours sans entrer dans le détail des algorithmes (c'est le sujet des notebooks suivants), regardons un modèle simple mais qui **apprend** vraiment quelque chose : le **k plus proches voisins** (`KNeighborsClassifier`).

**Intuition (en 30 secondes)** : pour classer une nouvelle fleur, le modèle cherche dans le jeu d'entraînement la fleur **la plus ressemblante** (avec `n_neighbors=1`), et copie sa classe. Plus ressemblante = la plus proche au sens des 4 mesures. C'est tout.

Ce qui nous intéresse ici n'est **pas** l'algorithme — c'est de voir que le pattern `fit / predict / score` est **strictement identique** à celui du DummyClassifier.

In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=1)

knn.fit(X_train, y_train)
score_knn = knn.score(X_test, y_test)
print(f"KNN (1 voisin) : {score_knn:.2%}")

KNN (1 voisin) : 100.00%


Le score est très au-dessus du plancher. Le modèle a **réellement appris** quelque chose de la géométrie des fleurs dans le jeu d'entraînement.

## `predict` vs `predict_proba`

Un classifieur sait faire deux choses :

- `.predict(X)` → la classe prédite (par exemple `0`, `1` ou `2`)
- `.predict_proba(X)` → les **probabilités** d'appartenir à chaque classe

Avoir les probabilités est utile quand on veut connaître la **confiance** du modèle, ou fixer un seuil de décision.

In [10]:
# Sur les 5 premières fleurs de test
print("Vraies classes :", y_test[:5])
print("predict        :", knn.predict(X_test[:5]))
print("predict_proba  :")
knn.predict_proba(X_test[:5])

Vraies classes : [1 0 2 1 1]
predict        : [1 0 2 1 1]
predict_proba  :


array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.]])

## Le contrat universel — ce qu'il faut retenir

Ce qu'on vient de faire avec `DummyClassifier` et `KNeighborsClassifier` est **strictement** la même séquence qu'on fera pour :

- un **arbre de décision** (`DecisionTreeClassifier`) — notebook suivant
- une **régression** (`LinearRegression`, `DecisionTreeRegressor`…)
- des **modèles ensemblistes** (`RandomForestClassifier`, `XGBClassifier`…)
- voire des **réseaux de neurones** (`MLPClassifier`)

```{admonition} 🎯 À retenir
:class: important
**Le contrat scikit-learn** :
1. **`X` est une matrice 2D** `(n_samples, n_features)`, **`y` un vecteur 1D** `(n_samples,)`.
2. **Toujours** découper en train/test (`train_test_split`) avant tout entraînement.
3. **Toujours** commencer par une baseline (`DummyClassifier` / `DummyRegressor`) pour savoir contre quoi se comparer.
4. Tous les modèles s'utilisent avec la **même triade** : `.fit(X_train, y_train)` → `.predict(X_test)` → `.score(X_test, y_test)`.
5. Pour la **confiance** d'un classifieur : `.predict_proba(X)`.
```

Avec ça en tête, on peut maintenant ouvrir la boîte noire d'un vrai algorithme — l'arbre de décision.